# **Exploratory Data Analysis with SQL**

## Introduction
In this notebook, I'll be:

1. Working with the SpaceX dataset to better understand patterns in launches
2. Loading the dataset into a SQLite database for analysis
3. Executing SQL queries to extract meaningful insights about SpaceX launches


In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
%cd "/content/drive/Othercomputers/My MacBook Pro/portfolio/spacex-data-science-capstone"
file_prefix = %pwd

/content/drive/Othercomputers/My MacBook Pro/portfolio/spacex-data-science-capstone


### Connect to the database

In [ ]:
# Install SQL extension if working locally
#!pip install ipython-sql

In [4]:
%load_ext sql

In [12]:
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [5]:
import csv, sqlite3

# Create a connection to the SQLite database
con = sqlite3.connect("my_data1.db")
cur = con.cursor()

In [6]:
# Connect to the SQLite database using SQL magic
%sql sqlite:///my_data1.db

In [7]:
# Load the SpaceX dataset and save it to the SQLite database
import pandas as pd
df = pd.read_csv("data/Spacex.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False, method="multi")

101

## SQL Analysis

Now I'll write and execute SQL queries to analyze the SpaceX launch data.

### Unique Launch Sites

In [13]:
%%sql
-- Query to find unique launch sites
SELECT DISTINCT Launch_Site
FROM SPACEXTBL;

 * sqlite:///my_data1.db
Done.


Launch_Site
CCAFS LC-40
VAFB SLC-4E
KSC LC-39A
CCAFS SLC-40


### Cape Canaveral Air Force Station Launches

Let's examine launches from sites that begin with "CCA" (Cape Canaveral Air Force Station).


In [14]:
%%sql
-- Find launches from Cape Canaveral Air Force Station
SELECT *
FROM SPACEXTBL
WHERE Launch_Site LIKE "CCA%"
LIMIT 5;

 * sqlite:///my_data1.db
Done.


Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### NASA Payload Mass

I'm interested in the total payload mass carried by boosters for NASA's Commercial Resupply Services missions.


In [15]:
%%sql
-- Calculate total payload mass for NASA (CRS) missions
SELECT SUM(PAYLOAD_MASS__KG_) AS Total_NASA_Payload_Mass
FROM SPACEXTBL
WHERE Customer = "NASA (CRS)";

 * sqlite:///my_data1.db
Done.


Total_NASA_Payload_Mass
45596


### Falcon 9 v1.1 Payload Capacity

Now I'll examine the average payload mass carried by the Falcon 9 v1.1 booster version.


In [16]:
%%sql
-- Count launches for each variant of the F9 v1.1 booster
SELECT Booster_Version, COUNT(*) AS Launch_Count
FROM SPACEXTBL
GROUP BY Booster_Version
HAVING Booster_Version LIKE "F9 v1.1%";

 * sqlite:///my_data1.db
Done.


Booster_Version,Launch_Count
F9 v1.1,5
F9 v1.1 B1003,1
F9 v1.1 B1010,1
F9 v1.1 B1011,1
F9 v1.1 B1012,1
F9 v1.1 B1013,1
F9 v1.1 B1014,1
F9 v1.1 B1015,1
F9 v1.1 B1016,1
F9 v1.1 B1017,1


In [17]:
%%sql
-- Calculate average payload mass for the base F9 v1.1 booster
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass
FROM SPACEXTBL
WHERE Booster_Version = "F9 v1.1";

 * sqlite:///my_data1.db
Done.


Avg_Payload_Mass
2928.4


In [18]:
%%sql
-- Calculate average payload mass for all F9 v1.1 variants
SELECT AVG(PAYLOAD_MASS__KG_) AS Avg_Payload_Mass_All_Variants
FROM SPACEXTBL
WHERE Booster_Version LIKE "F9 v1.1%";

 * sqlite:///my_data1.db
Done.


Avg_Payload_Mass_All_Variants
2534.6666666666665


### First Successful Ground Pad Landing

I want to identify when SpaceX achieved its first successful landing on a ground pad.


In [19]:
%%sql
-- Find the date of the first successful ground pad landing
SELECT Date
FROM SPACEXTBL
WHERE Landing_Outcome = "Success (ground pad)"
ORDER BY Date
LIMIT 1;

 * sqlite:///my_data1.db
Done.


Date
2015-12-22


### Successful Drone Ship Landings with Medium Payloads

Let's find boosters that successfully landed on drone ships while carrying payloads between 4000kg and 6000kg.


In [20]:
%%sql
-- Find boosters with successful drone ship landings and medium-weight payloads
SELECT DISTINCT Booster_Version
FROM SPACEXTBL
WHERE Landing_Outcome = "Success (drone ship)" AND PAYLOAD_MASS__KG_ BETWEEN 4000 AND 6000;

 * sqlite:///my_data1.db
Done.


Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### Mission Success and Failure Rates


In [21]:
%%sql
-- View all possible mission outcomes
SELECT DISTINCT Mission_Outcome
FROM SPACEXTBL;

 * sqlite:///my_data1.db
Done.


Mission_Outcome
Success
Failure (in flight)
Success (payload status unclear)
Success


In [22]:
%%sql
-- Count successful missions
SELECT COUNT(*) AS Total_Success
FROM SPACEXTBL
WHERE Mission_Outcome LIKE "%Success%";

 * sqlite:///my_data1.db
Done.


Total_Success
100


In [23]:
%%sql
-- Count failed missions
SELECT COUNT(*) AS Total_Failure
FROM SPACEXTBL
WHERE Mission_Outcome LIKE "%Failure%";

 * sqlite:///my_data1.db
Done.


Total_Failure
1


### Maximum Payload Capacity

Let's determine which booster versions have carried the maximum payload mass.


In [24]:
%%sql
-- Find boosters that carried the maximum payload mass using a subquery
SELECT Booster_Version, PAYLOAD_MASS__KG_
FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL);

 * sqlite:///my_data1.db
Done.


Booster_Version,PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


### Failed Drone Ship Landings in 2015

Now I'll analyze the failed drone ship landings in 2015, categorized by month.


In [25]:
%%sql
-- Analyze failed drone ship landings in 2015 by month
SELECT substr(Date, 6, 2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE substr(Date, 0, 5) = "2015" AND Landing_Outcome = "Failure (drone ship)";

 * sqlite:///my_data1.db
Done.


Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Landing Outcome Rankings

Finally, I'll rank the frequency of different landing outcomes between 2010-06-04 and 2017-03-20.


In [26]:
%%sql
-- Rank landing outcomes by count within a specific date range
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN "2010-06-04" AND "2017-03-20"
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;

 * sqlite:///my_data1.db
Done.


Landing_Outcome,Outcome_Count
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1


## Summary of Findings

Through this SQL analysis, I've gained several insights about SpaceX launches:

1. SpaceX utilizes multiple launch sites, with Cape Canaveral being particularly important
2. The company has conducted numerous missions for NASA's Commercial Resupply Services
3. Different booster versions have varying payload capacities
4. I've identified key milestones like the first successful ground pad landing
5. There's a clear progression in landing success rates over time
6. Certain boosters are capable of handling medium to heavy payloads while still landing successfully

These insights help build a foundation for predicting first stage landing success, which has significant economic implications given the cost savings associated with reusable rockets.